In [1]:
import shaker
import MDAnalysis as md
import nglview as nv

/local/lborge01/miniconda3/envs/general_env/lib/python3.12/site-packages/Bio/Application/__init__.py:39: BiopythonDeprecationWarning: The Bio.Application modules and modules relying on it have been deprecated.

Due to the on going maintenance burden of keeping command line application
wrappers up to date, we have decided to deprecate and eventually remove these
modules.

We instead now recommend building your command line and invoking it directly
with the subprocess module.
  warnings.warn(


## Generating type 3 virtual sites. This tutorial is a WIP.

Here we will use `shaker` to construct a [type 3 virtual-site](https://manual.gromacs.org/documentation/current/reference-manual/functions/interaction-methods.html#virtual-interaction-sites) representation that captures a molecules average structure. This greatly facilitates the parameterization of rigid molecules, such as cholesterol.

The workflow consists of the following steps:
* **Align all molecules in the trajectory to a common reference:**
  Text
* **Compute an average structure:**
  Text
* **Define a 3 bead frame:**
  Text
* **Construct the remaining beads as virtual sites:**
  The positions of the remaining beads are expressed as virtual sites whose coordinates depend on the frame beads.

In [3]:
author  = 'Luis Borges-Araujo'
itp_header = [
    f"; Parameterised by {author} @ ENS de Lyon, 2026.\n"
    "; Molecular name: \n",
    "; SMILES: \n",
]


## AA reference sims (already PBC treated and removed solvent.)
GRO = '../AA_references/ComplexMembrane/pbc.gro'
XTC = '../AA_references/ComplexMembrane/shortpbc.xtc'

mapping = {
    ## Resname
    "CHL1": {
    # Bead name;  Mapping.
        "ROH": {"type": "P1",    "charge": 0,  "atoms": ['C2', 'C2', 'C4', 'C4', 'O3', 'O3', 'O3', 'O3'] },      
        "R1":  {"type": "SC4",   "charge": 0,  "atoms": ['C6', 'C6', 'C6', 'C6','C6', 'C5', 'C7','H6','H7A','H7B'] },             
        "R2":  {"type": "SC3",   "charge": 0,  "atoms": ['C9', 'H9', 'C10', 'C1','H1B','H1A','C9', 'H9', 'C10', 'C1','H1B','H1A','H1B','H1A','H1A','H1B'] }, 
        "R3":  {"type": "SC3",   "charge": 0,  "atoms": ['C15', 'C15', 'C15', 'C15', 'C15', 'C14','C14','C16','C16'] },      
        "R4":  {"type": "SC3",   "charge": 0,  "atoms": ['C11', 'C12'] }, 
        "R5":  {"type": "TC2",   "charge": 0,  "atoms": ['C19'] },              
        "R6":  {"type": "TC2",   "charge": 0,  "atoms": ['C18'] },              
        "C1":  {"type": "C2",    "charge": 0,  "atoms": ['C20','H22A','H22B','H20','H21A','H21B','H21C'] },
        "C2":  {"type": "C2",    "charge": 0,  "atoms": ['C23', 'C24', 'C25', 'C26', 'C27'] },
    },
}

shaker.mapper.map_aa2cg(GRO, XTC, mapping)

  0%|          | 0/51 [00:00<?, ?it/s]

/local/lborge01/gitbuilds/Shaker/shaker/mapper.py:124: DuplicateWarning: AtomGroup.center_of_geometry(): 'ag' <AtomGroup with 8 atoms> contains duplicates. Results might be biased!
  coords[k] = ag.center_of_geometry()
/local/lborge01/gitbuilds/Shaker/shaker/mapper.py:124: DuplicateWarning: AtomGroup.center_of_geometry(): 'ag' <AtomGroup with 10 atoms> contains duplicates. Results might be biased!
  coords[k] = ag.center_of_geometry()
/local/lborge01/gitbuilds/Shaker/shaker/mapper.py:124: DuplicateWarning: AtomGroup.center_of_geometry(): 'ag' <AtomGroup with 16 atoms> contains duplicates. Results might be biased!
  coords[k] = ag.center_of_geometry()
/local/lborge01/gitbuilds/Shaker/shaker/mapper.py:124: DuplicateWarning: AtomGroup.center_of_geometry(): 'ag' <AtomGroup with 9 atoms> contains duplicates. Results might be biased!
  coords[k] = ag.center_of_geometry()
/local/lborge01/miniconda3/envs/general_env/lib/python3.12/site-packages/MDAnalysis/coordinates/GRO.py:479: UserWarning: m

In [4]:
shaker.vsites.align_mol_to_single_traj('cg_mapped.gro', 'cg_mapped.xtc',
                    selection="resname CHOL CHL1 SITO ERG CAMP STIG",
                    align_selection="name C1 R2 R1",
                    reference_residue=0,)

/local/lborge01/miniconda3/envs/general_env/lib/python3.12/site-packages/MDAnalysis/topology/guessers.py:146: UserWarning: Failed to guess the mass for the following atom types: R
  warnings.warn("Failed to guess the mass for the following atom types: {}".format(atom_type))
/local/lborge01/miniconda3/envs/general_env/lib/python3.12/site-packages/MDAnalysis/coordinates/XDR.py:240: UserWarning: Reload offsets from trajectory
 ctime or size or n_atoms did not match
  warnings.warn("Reload offsets from trajectory\n "


Aligning molecules:   0%|          | 0/51 [00:00<?, ?it/s]

Averaging aligned trajectory:   0%|          | 0/3978 [00:00<?, ?it/s]

In [5]:
view = nv.show_mdanalysis(md.Universe('cg_mapped.gro','cg_mapped.xtc'))
view.clear_representations()
view.add_representation("spacefill", selection="all", radius=2.5)
view

NGLWidget(max_frame=50)

In [6]:

view = nv.show_mdanalysis(md.Universe('average_molecule.gro', 'aligned_molecules.xtc'))
view.clear_representations()
view.add_representation("spacefill", selection="all", radius=2.5)
view

/local/lborge01/miniconda3/envs/general_env/lib/python3.12/site-packages/MDAnalysis/topology/guessers.py:146: UserWarning: Failed to guess the mass for the following atom types: R
  warnings.warn("Failed to guess the mass for the following atom types: {}".format(atom_type))


NGLWidget(max_frame=3977)

In [7]:
u=md.Universe('average_molecule.gro')

lines, mapping = shaker.vsites.generate_virtual_sites3(u, ['C1','R2',"R1"], output='index', selection='not name C2', 
                                         mass_split="equal", mapping=mapping, resname='CHL1')

/local/lborge01/miniconda3/envs/general_env/lib/python3.12/site-packages/MDAnalysis/topology/guessers.py:146: UserWarning: Failed to guess the mass for the following atom types: R
  warnings.warn("Failed to guess the mass for the following atom types: {}".format(atom_type))
/local/lborge01/miniconda3/envs/general_env/lib/python3.12/site-packages/MDAnalysis/coordinates/GRO.py:228: UserWarning: Empty box [0., 0., 0.] found - treating as missing unit cell. Dimensions set to `None`.
  warnings.warn(wmsg)


In [8]:
lines

['[ constraints ]',
 '     8      3   1  0.74997',
 '     8      2   1  0.78758',
 '     3      2   1  0.34851',
 '',
 '[ virtual_sites3 ]',
 '     1   8  3  2   4  1.09190  0.35434  0.23571',
 '     4   8  3  2   3  -0.41527  0.83936',
 '     5   8  3  2   3  0.69152  -0.10187',
 '     6   8  3  2   4  0.78256  0.21689  0.93342',
 '     7   8  3  2   4  0.06513  0.32174  0.77189',
 '',
 '[ exclusions ]',
 '     1   2  3  4  5  6  7  8',
 '     2   3  4  5  6  7  8',
 '     3   4  5  6  7  8',
 '     4   5  6  7  8',
 '     5   6  7  8',
 '     6   7  8',
 '     7   8',
 '']

In [9]:
shaker.itp.write_initial_CGitp("CHL1", mapping,
                               header = itp_header,
                               footer=lines)